# Phugoid Motion

In the first module of **Practical Numerical Methods** we study problems modeled by ordinary differential equations.

Our motivating problem is the _phugoid model of glider flight_. In this first lesson, we discuss some background, explain the physics, and work out the mathematical model.

We first consider the idealized model with no drag, resulting in a simple harmonic motion. We will plot some interesting trajectories of a glider under phugoid oscillations under the ideal model. In the next lesson, you'll learn to numerically integrate the differential equation using Euler's method.

:::{note} The lesson pattern
:icon: false

This is our first complete pass through a practice that will repeat throughout the course:

1. **Derive — On paper:** establish an expectation independently of the computer.
2. **Reconstruct — In your notebook:** rebuild the central calculation from the worked lesson.
3. **Specify — Before using an agent:** state what is being delegated and what evidence would make its result acceptable.
4. **Audit — Review and verify:** inspect the proposal and test whether the evidence can expose a plausible defect.
5. **Explain — Your verdict:** decide what the evidence supports and what remains uncertain.

The **specification** step may be unfamiliar. It is not merely the wording of a prompt: it records the outcome, model context, interface, constraints, permitted access, and acceptance evidence that you have decided before delegating work. This lesson provides a detailed scaffold; later lessons will ask you to make more of these decisions yourself.
:::

The term _phugoid_ in aeronautics refers to a motion pattern where an aircraft oscillates up and down —nose-up and climb, then nose-down and descend— around an equilibrium trajectory.  The aircraft oscillates in altitude, speed and pitch, with only small (neglected) variations in the angle of attack, as it repeatedly exchanges kinetic and potential energy [@sinha2013, chap. 5].

A low-amplitude phugoid motion can be just a nuisance, as the aircraft does not exceed the stall angle of attack and nothing bad happens. But the mode can also be unstable leading to a stall or even a loop.

Look at this [video](https://www.youtube.com/watch?v=ysdU4mnRYdM) of a simulator showing a Cessna single-engine airplane in phugoid motion:

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo('ysdU4mnRYdM')

That doesn't look too good! What's happening? 

It can get a lot worse when an aircraft enters one of these modes that is unstable. A famous example is that of [NASA's Helios Solar Powered Aircraft](https://www.nasa.gov/image-article/helios-prototype-flying-wing-29/) prototype, which broke up in mid air due to extreme phugoid oscillations.

Helios was a proof-of-concept solar electric-powered flying wing that broke the world altitude record for a non-rocket-powered aircraft in August 2001. But in June 26, 2003, it broke something else. The aircraft entered phugoid motion after encountering turbulence near the Hawaiian Island of Kauai. The high speed reached in the oscillations exceeded the design limits, and it ended up wrecked in the Pacific Ocean. Luckily, the Helios was remotely operated, and nobody got hurt.

## The physics of phugoids

In a phugoid oscillation the aircraft pitches up and down, as it decelerates and accelerates. The trajectory might look like a sinusoid, as shown in [](#fig-phugoid-trajectory). The assumption is that the forward velocity of the aircraft, $v$, varies in such a way that the angle of attack remains (nearly) constant, which means that we can assume a constant lift coefficient.

```{figure} ./figures/oscillatory_trajectory.png
:label: fig-phugoid-trajectory
:alt: A sinusoidal trajectory showing an aircraft oscillating above and below a central flight path
:align: center

Trajectory of an aircraft in phugoid motion.
```

In the descending portion of the trajectory, the aircraft's velocity increases as it proceeds from a peak to the minimum height—gaining kinetic energy at the expense of potential energy. The contrary happens in the upward segment, as its velocity decreases there.

We measure the pitch angle (between the aircraft's longitudinal axis and the horizontal) as positive when the aircraft's nose is pointing up. In the portion of the trajectory below the center-line, where it curves upwards, the pitch angle $\theta$ is increasing: $\dot{\theta}>0$. And where the trajectory curves down, the pitch angle is decreasing: $\dot{\theta}<0$, as shown in [](#fig-phugoid-trajectory).

Let's review the forces affecting an aircraft in a downward glide. In [](#fig-glider-forces), we show the flight path, the forces on the glider (no thrust), and the _glide angle_ or flight path angle, $\gamma$, between the flight path and the horizontal.

```{figure} ./figures/glider_forces.png
:label: fig-glider-forces
:alt: Lift, drag, and weight acting on a glider along a descending flight path
:align: center

Forces on a glider.
```

The force of lift, $L$ —created by the airflow around the wings— is perpendicular to the trajectory, and the force of drag, $D$, is parallel to the trajectory. Both forces are expressed in terms of coefficients of lift and drag, $C_L$ and $C_D$, respectively, that depend on the wing design and _angle of attack_: the angle between the wing chord and the flight path.

If you are not familiar with airplane aerodynamics, you might be getting confused with some terms here ... and all those angles! But be patient and look things up, if you need to. We're giving you a quick summary here.

Lift and drag are proportional to a surface area, $S$, and to the dynamic pressure: $1/2 \rho v^2$, where $\rho$ is the density of air, and $v$ the forward velocity of the aircraft. The equations for lift and drag are:

$$
\label{eq-lift}
L = C_L S \times \frac{1}{2} \rho v^2
$$

$$
\label{eq-drag}
D = C_D S \times \frac{1}{2} \rho v^2
$$

When the glider is in equilibrium, the forces balance each other. We can equate the forces in the directions perpendicular and parallel to the trajectory, as follows:

$$
\begin{equation}
L = W \cos \gamma \quad \text{and} \quad D = W \sin \gamma
\end{equation}
$$

where $W$ represents the weight of the glider.

In the figure, we've drawn the angle $\gamma$ as the _glide angle_, formed between the direction of motion and the horizontal. We are not bothered with the _sign_ of the angle, because we draw a free-body diagram and take the direction of the forces into account in writing our balance equations. But later on, we will need to be careful with the sign of the angles. It can cause you a real headache to keep this straight, so be patient!

Below, we will develop the mathematical model step-by-step. But before, a short glimpse of the history.

## Lanchester's Aerodonetics

"Phugoid theory" was first described by the British engineer Frederick W. Lanchester in _Aerodonetics_ [@lanchester1908]. The book is now in the public domain, so you can [read or download it from the Internet Archive](https://archive.org/details/aerodoneticscon02lancgoog/page/n7/mode/2up).

Lanchester defines phugoid theory as the study of longitudinal stability of a flying machine (aerodone). He first considered the simplification where drag and moment of inertia are neglected. Then he included these effects, obtaining an equation of stability. In addition to describing many experiments by himself and others, Lanchester also reports on "numerical work ... done by the aid of an ordinary 25-cm slide rule" [@lanchester1908]. Go figure!

### Ideal case of zero drag

In this section, we follow the derivation given by @milnethomson1966 [sec. 18.5], which we find a little bit easier than that of the original in "Aerodonetics."

An aircraft flying in steady, straight horizontal flight has a lift equal to its weight. The velocity in this condition is sometimes called _trim velocity_ ("trim" is what pilots do to set the controls to just stay in a steady flight). Let's use $v_t$ for the trim velocity, and from $L=W$ deduce that:

$$
\begin{equation}
W = C_L S \times\frac{1}{2} \rho v_t^2
\end{equation}
$$

The weight $W$ is constant for the aircraft, but the lift at any other flight condition depends on the flight speed, $v$. We can use the expression for the weight in terms of $v_t$ to obtain the ratio $L/W$ at any other flight velocity, as follows:

$$
\begin{equation}
\frac{L}{W}= \frac{v^2}{v_t^2}
\end{equation}
$$

Imagine that the aircraft experienced a little upset, a wind gust, and it finds itself off the "trim" level, in a curved path with an instantaneous angle $\theta$. In [](#fig-glider-curved-trajectory), we exaggerate the curved trajectory of flight to help you visualize what we'll do next. The angle $\theta$ (using the same name as Milne-Thompson) is between the _trajectory_ and the horizontal, positive up.

```{figure} ./figures/glider_forces_nodrag.svg
:label: fig-glider-curved-trajectory
:alt: Curved upward trajectory of an aircraft with lift and weight vectors
:width: 600px
:align: center

Curved trajectory of an aircraft climbing with no drag.
```

```{figure} ./figures/glider_forces_fbd.svg
:label: fig-glider-free-body
:alt: Free-body diagram showing lift, weight, and normal and tangential directions
:width: 300px
:align: center

Free-body diagram of the aircraft trajectory.
```

From [](#fig-glider-free-body), we can see that

$$
\begin{equation}
\vec{L} + \vec{W} = m\vec{a} = \frac{mv^2}{R}\hat{n} + m \frac{dv}{dt}\hat{t}
\end{equation}
$$

where $\frac{v^2}{R}$ is the centripetal acceleration and $R$ is the radius of curvature of the trajectory.
If we decompose the lift and weight into their normal and tangential components we get

$$
\begin{equation}
L\hat{n} + W_n\hat{n} + W_t\hat{t} = \frac{mv^2}{R}\hat{n} + m \frac{dv}{dt}\hat{t}
\end{equation}
$$

The component of the weight in the normal direction ($W_n$) is

$$
\begin{equation}
W_n = -W \cos \theta
\end{equation}
$$

If we then consider that all of the components in $\hat{n}$ must balance out, we arrive at

$$
\begin{equation}
L - W \cos \theta = \frac{mv^2}{R}
\end{equation}
$$

We can rewrite this as

$$
\begin{equation}
L- W \cos \theta = \frac{W}{g} \frac{v^2}{R}
\end{equation}
$$

where $g$ is the acceleration due to gravity. Rearrange this by dividing the equation by the weight, and use the expression we found for $L/W$, above. The following equation results:

$$
\begin{equation}
\frac{v^2}{v_t^2}-\cos \theta = \frac{v^2}{g R}
\end{equation}
$$

Recall that we simplified the problem assuming that there is no friction, which means that the total energy is constant (the lift does no work). If $z$ represents the depth below a reference horizontal line, the energy per unit mass is (kinetic plus potential energy):

$$
\begin{equation}
\frac{1}{2}v^2-g z = \text{constant}
\end{equation}
$$

To get rid of that pesky constant, we can choose the reference horizontal line at the level that makes the constant energy equal to zero, so $v^2 = 2 g z$. That helps us re-write the phugoid equation in terms of $z$ as follows:

$$
\begin{equation}
\frac{z}{z_t}-\cos \theta = \frac{2z}{R}
\end{equation}
$$

Let $ds$ represent a small arc-length of the trajectory. We can write 

$$
\begin{equation}
\frac{1}{R} = \frac{d\theta}{ds} \quad \text{and}\quad  \sin\theta = -\frac{dz}{ds}
\end{equation}
$$

Employing the chain rule of calculus,

$$
\label{eq-curvature-chain-rule}
\frac{1}{R} = \frac{d\theta}{ds} = \frac{dz}{ds}\frac{d\theta}{dz} = -\sin \theta\frac{d\theta}{dz}
$$

Multiply the phugoid equation by $\frac{1}{2\sqrt{z}}$ to get:

$$
\begin{equation}
\frac{\sqrt{z}}{2z_t} - \frac{\cos\theta}{2\sqrt{z}} = \frac{\sqrt{z}}{R}
\end{equation}
$$

Substituting for $1/R$ on the right hand side and bringing the cosine term over to the right, we get:

$$
\begin{equation}
\frac{\sqrt{z}}{2z_t} = \frac{\cos \theta}{2 \sqrt{z}} - \sqrt{z} \sin \theta \frac{d\theta}{dz}
\end{equation}
$$

The right-hand-side is an exact derivative! We can rewrite it as:

$$
\begin{equation}
\frac{d}{dz} \left(\sqrt{z}\cos\theta \right) = \frac{\sqrt{z}}{2z_t}
\end{equation}
$$

Integrating this equation, we add an arbitrary constant, chosen as $C\sqrt{z_t}$ which (after dividing through by $\sqrt{z}$) gives:

$$
\label{eq-phugoid-curve}
\cos \theta = \frac{1}{3}\frac{z}{z_t} + C\sqrt{\frac{z_t}{z}}
$$

Taking the derivative of both sides of [Equation %s](#eq-phugoid-curve) and applying the relation in [Equation %s](#eq-curvature-chain-rule) yields:

$$
\label{eq-radius-of-curvature}
\frac{z_t}{R} = \frac{1}{3} - \frac{C}{2}\sqrt{\frac{z_t^3}{z^3}}
$$

:::{warning .simple .dropdown icon=false open=false} On paper

Before using code, reproduce the pivotal part of the derivation by hand:

1. Starting from [Equation %s](#eq-phugoid-curve), differentiate with respect to $z$ and use [Equation %s](#eq-curvature-chain-rule) to obtain [Equation %s](#eq-radius-of-curvature).
2. Check that $C$ and $z_t/R$ are dimensionless.
3. For $z_t=64$, $z_0=16$, and $\theta_0=0$, calculate $C$ and predict the sign of the initial radius of curvature $R$.
4. Sketch which way the trajectory should initially bend.

Keep this work beside you. Its values and predictions are independent evidence for checking the computation that follows. See [Before computing](../../appendices/verification-patterns.md#verification-before-computing) for the general role of these checks.
:::

### Phugoid Curves

[Equation %s](#eq-phugoid-curve) is nonlinear, so we are hard-pressed to write a clean expression for the complete path $z(x)$. Lanchester said that he was unable to _"reduce this expression to a form suitable for co-ordinate plotting."_ Instead, he devised what he called the "trammel" method. His explanation begins on page 46 of _Aerodonetics_ [@lanchester1908, p. 46].

Lanchester used [Equation %s](#eq-phugoid-curve) to calculate the integration constant $C$ and [Equation %s](#eq-radius-of-curvature) to calculate the local radius of curvature $R$, then constructed the trajectory as a succession of small circular arcs—by hand.

In the next section, we reproduce that process computationally. **We will use this exercise explicitly as a Python refresher** while we explore the simplest, zero-drag phugoid model.

If Python is new to you, first work through [Python essentials for this course](../../appendices/python-essentials.ipynb), then keep it open as a reference. If your Python is rusty, take time with the reminders and reconstruct one small step at a time in your own notebook. If you are already an experienced programmer, focus on how the mathematics becomes a small set of explicit functions, how computation is separated from plotting, and where we check the model's domain.

The computational idea is simple: start from an initial point and angle, take a short step of arc length $ds$, calculate the local circle from $R$, rotate the point around that circle, and repeat.

## Computing trajectories: a Python refresher

:::{warning .simple .dropdown icon=false open=false} In your notebook

Create a new notebook for your work, rather than simply executing the code in this lesson notebook. Keep the lesson open as a reference and reconstruct the trammel calculation there.

Before running your first calculation, record the value of $C$, the sign of $R$, and the initial direction you predicted on paper. On your work notebook, type the equations, geometric update, loop, and model checks yourself. You may copy small mechanical elements such as imports or plot labels when transcription would add no understanding.

As you proceed, compare each computed result with your prediction and explain any disagreement before continuing. Consult [Reconstruct a lesson](../../appendices/notebook-workflow.md#notebook-reconstruct) for the general workflow.
:::

We need two widely used scientific-Python packages: [NumPy](http://numpy.org) for numerical functions and arrays, and [Matplotlib](http://matplotlib.org) for plotting.

:::{note} Python refresher — imports and aliases
:icon: false

An `import` statement makes code from another package available in the notebook. The names `np` and `plt` are conventional aliases: they save typing and make the origin of calls such as `np.cos()` and `plt.subplots()` immediately recognizable. An alias is only a name; it does not change the package.
:::

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Translate the equations into functions

First, solve [Equation %s](#eq-phugoid-curve) for $C$ at a known point $(z,\theta)$:

$$
C = \left(\cos\theta - \frac{z}{3z_t}\right)\sqrt{\frac{z}{z_t}}.
$$

[Equation %s](#eq-radius-of-curvature) gives the corresponding signed radius of curvature:

$$
R = \frac{z_t}{\frac{1}{3} - \frac{C}{2}\left(\frac{z_t}{z}\right)^{3/2}}.
$$

The Python code below translates those equations almost literally. Read the code carefully and convince yourself that it does.

:::{note} Python refresher — defining functions
:icon: false

The `def` statement names a reusable calculation. Its parameters are local names for the input values, and `return` sends the result back to the caller. Python uses indentation (not braces) to mark the function body. We use `snake_case` names and underscores in `z_t` so the code resembles the mathematical notation.
:::

Both $z$ and $z_t$ represent positive depths. We check that assumption explicitly. A zero denominator in the second equation means a straight path with infinite radius, represented by `np.inf`.

:::{note} Python refresher — error handling
:icon: false

The `raise` keyword is used for error handling to stop execution when given invalid inputs. It immediately stops the function, preventing the invalid values from reaching the math calculation below. Here, it triggers a specific type of error called a `ValueError`, and it also attaches the custom message `"z and z_t must be positive."` to the error.
:::

In [ ]:
def integration_constant(z, z_t, theta):
    '''Return the integration constant C; theta is in radians.'''
    if z <= 0.0 or z_t <= 0.0:
        raise ValueError("z and z_t must be positive.")
    return (np.cos(theta) - z / (3.0 * z_t)) * np.sqrt(z / z_t)


def radius_of_curvature(z, z_t, C):
    '''Return the signed local radius of curvature.'''
    if z <= 0.0 or z_t <= 0.0:
        raise ValueError("z and z_t must be positive.")
    denominator = 1.0 / 3.0 - C / 2.0 * (z_t / z)**1.5
    if np.isclose(denominator, 0.0):
        return np.inf
    return z_t / denominator

Let's evaluate the two functions at one initial condition. Angles may be convenient to specify in degrees, but NumPy's trigonometric functions expect radians, so we convert at the boundary between user input and computation.

:::{note} Python refresher — assignment, calls, and checks
:icon: false

The `=` operator binds a value to a name (called an "assignment"). Parentheses after a function name call it. An f-string—marked by the `f` before the opening quote—places values inside text and can control their formatting. The chained comparison in the `assert` statement is a compact, Pythonic way to check an expectation. Assertions are useful while exploring, although a reusable module would eventually need a proper test suite.
:::

A signed $R$ records which side of the trajectory contains the center of curvature; its magnitude is the geometric radius.

In [ ]:
z_t = 64.0
z_0 = 16.0
theta_0_degrees = 0.0
theta_0 = np.deg2rad(theta_0_degrees)

C = integration_constant(z_0, z_t, theta_0)
R_0 = radius_of_curvature(z_0, z_t, C)

print(f"C = {C:.3f}")
print(f"Initial signed radius R = {R_0:.3f}")
assert 0.0 < C < 2.0 / 3.0
np.testing.assert_allclose(C, 11.0 / 24.0, rtol=0.0, atol=1e-12)
np.testing.assert_allclose(R_0, -128.0 / 3.0, rtol=0.0, atol=1e-12)

**Reconstruction checkpoint.** The two `assert_allclose()` calls connect the computation to the values obtained independently on paper: $C=11/24$ and $R_0=-128/3$. Do not continue until the value, sign, and predicted initial bend agree. If they do not, inspect the derivative, angle units, argument order, and sign convention before changing the expected values.

This is the first handoff in our lesson pattern: the derivation has supplied acceptance evidence for the reconstructed code.

### Rotate one point around a local circle

To follow one short circular arc, we rotate the current point $(x,z)$ about the center of curvature. The formula below is a standard two-dimensional rotation written for the sign convention used in our diagram.

:::{note} Python refresher — tuples and unpacking
:icon: false

A pair such as `(x_center, z_center)` is a tuple. The statement `x_center, z_center = center` unpacks its two elements into two names. A function can likewise appear to return two values; Python packages them into a tuple, which the caller can unpack.
:::

In [ ]:
def rotate_point(x, z, center, angle):
    '''Rotate the point (x, z) about center by angle radians.'''
    x_center, z_center = center
    dx, dz = x - x_center, z - z_center

    x_new = x_center + dx * np.cos(angle) + dz * np.sin(angle)
    z_new = z_center - dx * np.sin(angle) + dz * np.cos(angle)
    return x_new, z_new

### March along the trajectory

We now have the pieces needed to imitate Lanchester's trammel. The function below repeatedly calculates $R$, locates the center of curvature, and rotates the current point through the small angle $d\theta=ds/R$.

:::{note} Python refresher — defaults, lists, loops, and branches
:icon: false

Parameters with values in the function definition are optional arguments with defaults. We store coordinates in Python lists because the trajectory may stop at the boundary $z=0$ before using every requested step. The index `-1` means the last item in a sequence, while `.append()` adds a new item. A `for` loop repeats its indented body; `_` is the conventional name when we do not otherwise need the loop counter. `if`, `else`, and `break` let the algorithm choose a path or stop. Finally, `np.asarray()` converts the completed lists to NumPy arrays for numerical work and plotting.
:::

Notice two model-aware branches. An infinite radius produces a straight segment. Also, the ideal formula assumes positive depth $z$, so we stop the trace before $z$ becomes non-positive.

Read this code carefully and convince yourself that it matches our model.

In [ ]:
def trace_phugoid(z_t, z_0, theta_0, n_steps=1000, ds=1.0):
    '''Trace a zero-drag phugoid; theta_0 is supplied in degrees.'''
    if n_steps < 2:
        raise ValueError("n_steps must be at least 2.")

    theta = np.deg2rad(theta_0)
    C = integration_constant(z_0, z_t, theta)

    x_values = [0.0]
    z_values = [z_0]

    for _ in range(n_steps - 1):
        x_current, z_current = x_values[-1], z_values[-1]
        R = radius_of_curvature(z_current, z_t, C)

        if np.isinf(R):
            dtheta = 0.0
            x_new = x_current + ds * np.cos(theta)
            z_new = z_current - ds * np.sin(theta)
        else:
            normal = np.array([-np.sin(theta), -np.cos(theta)])
            center = np.array([x_current, z_current]) + R * normal
            dtheta = ds / R
            x_new, z_new = rotate_point(
                x_current, z_current, center, dtheta
            )

        if z_new <= 0.0:
            break

        x_values.append(x_new)
        z_values.append(z_new)
        theta += dtheta

    return np.asarray(x_values), np.asarray(z_values), C

### Keep plotting separate from computation

The trajectory function returns numerical data without deciding how it must be displayed. A second, smaller function handles the plot. This separation lets us inspect, test, or reuse the coordinates without producing a figure every time.

:::{note} Python refresher — objects and methods
:icon: false

[`plt.subplots()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html) returns a Figure object and an Axes object. Calls such as `ax.plot()` and `ax.set_xlabel()` are methods: functions attached to an object. Keeping the `ax` name gives us explicit control of which plot we are modifying. The final `return fig, ax` also lets later code customize the figure.
:::

Our coordinate $z$ is depth measured positive downward, so the plot uses $-z$ to show upward displacement in the familiar direction.

In [ ]:
def plot_flight_path(x, z, C):
    '''Plot a previously computed phugoid trajectory.'''
    fig, ax = plt.subplots(figsize=(9.0, 4.0))
    
    ax.plot(x, -z, linewidth=2.0)
    ax.set_title(f"Flight path for C = {C:.3f}")
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$-z$")
    ax.grid()
    ax.set_aspect("equal", adjustable="box")
    return fig, ax

### Explore the family of phugoid curves

Look again at [Equation %s](#eq-phugoid-curve). The value of $C$ organizes the possible paths:

- $C>2/3$ gives no physical solution because it would require $\cos\theta>1$.
- $C=2/3$ gives the horizontal straight path: $\theta=0$ and $R=\infty$.
- $0<C<2/3$ gives trochoidal-like paths.
- $C<0$ can give paths containing loops.
- $C=0$ makes $R=3z_t$, a constant.

First choose conditions that give $0<C<2/3$. Before running the cell, predict the sign of the initial curvature and the general shape of the path.

:::{note} Python refresher — multiple assignment
:icon: false

`trace_phugoid()` returns three values in a tuple. The statement on the left unpacks them into three names. We omit `n_steps` and `ds`, so Python uses their default values.
:::

In [ ]:
x, z, C = trace_phugoid(z_t=64.0, z_0=16.0, theta_0=0.0)
fig, ax = plot_flight_path(x, z, C)

The calculated value is $C\approx0.458$, between $0$ and $2/3$, and the path is trochoidal as predicted.

Now keep the two depths fixed but reverse the initial direction to $180^\circ$. What sign do you predict for $C$? What new feature should appear in the path?

In [ ]:
x, z, C = trace_phugoid(z_t=64.0, z_0=16.0, theta_0=180.0)
fig, ax = plot_flight_path(x, z, C)

The negative value of $C$ produces loops. Experiment by changing one input at a time and predicting the result before you run the cell again.

:::{note} Python refresher — errors are information
:icon: false

A non-positive depth or too few requested steps raises a `ValueError` with an explanation. A clear failure helps us discover that an assumption or input is invalid.
:::

Notice that `trace_phugoid()` does not accept $C$ as an input; it derives $C$ from $z_0$, $z_t$, and $\theta_0$. The statement that $C>2/3$ is unphysical does not by itself establish that the tracer needs a separate runtime check for it. In the agent activity below, you will determine whether valid public inputs can produce that value and decide what belongs in input validation, mathematical reasoning, and executable tests.

For $C=0$, [Equation %s](#eq-radius-of-curvature) reduces to

$$
R=3z_t.
$$

The radius is constant. Setting $z_0=3z_t$ and $\theta_0=0$ forces $C=0$. Our ideal model is defined only for $z>0$, so the tracer follows the circular arc until just before it reaches the reference level.

In [ ]:
x, z, C = trace_phugoid(z_t=16.0, z_0=48.0, theta_0=0.0)
fig, ax = plot_flight_path(x, z, C)

The result is a quarter-circle arc with radius $48=3z_t$. We can obtain an almost semicircular path from another configuration where $C$ is close to zero:

In [ ]:
x, z, C = trace_phugoid(z_t=64.0, z_0=16.0, theta_0=-90.0)
fig, ax = plot_flight_path(x, z, C)

## Specify and audit validation with an agent

:::{warning .simple .dropdown icon=false open=true} With an agent — write the specification first

Use an agent to help evaluate your reconstructed `trace_phugoid()` function—not to replace it. **Do not invoke the agent yet.** First write a task specification in your own notebook. The specification records what you have decided and establishes the boundary of what you are delegating. It will also give you a standard against which to review the response.

For this first specification, complete each field below. Write it in your own words and add any assumptions that your implementation requires.

- **Outcome:** Propose executable validation checks for the reconstructed tracer and explain the purpose of each check.
- **Object under review:** `trace_phugoid(z_t, z_0, theta_0, n_steps=1000, ds=1.0)`. The implementation is already written and is not to be rewritten or optimized.
- **Model context:** Record the equation for signed $R$, the requirement $z,z_t>0$, the angle units at the interface, and the distinction between internal depth $z$ and plotted height $-z$.
- **Interface:** State the meaning of the five inputs, the returned `x`, `z`, and `C`, and the failure behavior you expect.
- **Constraints:** Use only NumPy; keep validation separate from the tracer; do not add dependencies, use the network, suppress warnings, or change the implementation.
- **Acceptance evidence:** Address the horizontal limit, the $C=0$ circular case including direction of travel, non-positive depths, too few steps, and the mathematical question of whether valid real initial conditions can produce $C>2/3$. For every proposed check, require a mathematical property, expected result, justified tolerance, and plausible defect it could expose.
- **Access:** Give the agent only the relevant functions or read-only access to your personal notebook. Do not grant write access for a validation proposal.
- **Open questions:** List assumptions or requested claims whose testability is not yet clear. Ask the agent to surface ambiguities before acting.

Before proceeding, point to each acceptance requirement and explain why it matters. See [Define the task by specification](../../appendices/agent-use.md#agent-task-specification) for the general pattern.
:::

### Turn the specification into an agent request

The specification is the standard the result must meet. The prompt below asks the agent to work against that standard. Use it only after you have written your specification, then append the specification and the relevant functions.

```text
Review my reconstructed trace_phugoid() function against the specification
below. Do not rewrite the function.

First, restate the claims that can be checked through its public inputs and
outputs, and identify any requested claim that is not directly testable.
Then propose NumPy-only validation functions for:

1. the horizontal limiting case;
2. the C = 0 circular case, including direction of travel;
3. non-positive depths and too few requested steps; and
4. the mathematical bound on C for positive depths and real angles.

For every executable check, name the property being tested, the expected
result, a reasonable tolerance, and one plausible defect it could expose.
Do not invent an input merely to satisfy a requested case. If a case is
mathematically impossible, explain why and state what evidence supports that
conclusion instead. Keep the checks separate from the implementation, and do
not use the network, add
dependencies, alter files, or claim that passing tests proves general
correctness.

[Paste your specification and the relevant functions here.]
```

A specification and a prompt have different roles. If the agent proposes a polished check that violates the specification, the check is still inadmissible. If the agent identifies a contradiction or an untestable claim, revise the specification explicitly rather than silently changing the task.

### Review the proposal before running it

Agent output is a proposal under review. Do not run candidate code until you have connected each check to a claim from your specification. Copy this table into your notebook and complete one row for every proposed check:

| Proposed check | Mathematical claim | Expected result and tolerance | Plausible defect exposed | Could wrong code still pass? | Decision |
| --- | --- | --- | --- | --- | --- |
|  |  |  |  |  | Accept, revise, or reject |

One acceptance requirement needs mathematical review before it can become an executable test. Let $r=z/z_t>0$ and rewrite the integration constant as

$$
C = \left(\cos\theta-\frac{r}{3}\right)\sqrt{r}.
$$

Use $\cos\theta\leq 1$ to find the largest value possible over positive $r$. If valid public inputs cannot produce $C>2/3$, record that finding as a revision to the requested runtime check: it is a mathematical bound to justify, not an input case to fabricate. Then explain why `trace_phugoid()` does not need a `C>2/3` input guard when it computes $C$ from those inputs.

### Audit the checks through defect injection

A check is useful only to the extent that it can disagree with a plausible wrong result. After reviewing and revising the proposal:

1. Preserve your working tracer unchanged.
2. Organize each validation function to accept the tracer as an argument. This lets the same evidence examine two implementations.
3. Make a temporary, clearly named copy of the tracer and reverse only the sign of `dtheta` in its circular update.
4. Run the accepted checks against both the working and deliberately defective versions.
5. Record which checks fail, which pass, and what each result means.

The identity $x^2+z^2=(3z_t)^2$ can pass for motion in either direction around the circle. For the stated circular initial condition, conditions such as `x[1] > x[0]` and `z[1] < z[0]` distinguish the intended orientation. If the agent omitted direction, revise the validation suite and record the omission. This combines [comparison with known behavior](../../appendices/verification-patterns.md#verification-known-behavior) and [deliberate defect injection](../../appendices/verification-patterns.md#verification-defect-injection).

If no agent is available, exchange specifications and validation proposals with a peer, or use an instructor-supplied candidate response. The specification, review, defect injection, and verdict remain the same.

We have reproduced trajectories that Lanchester painstakingly constructed by hand with his trammel. Along the way, we reviewed Python imports, function definitions, assignments, tuples, lists, arrays, loops, conditionals, exceptions, formatted strings, and object-oriented plotting.

More importantly, every code element corresponds to a modeling choice: the sign convention, the domain $z>0$, the local radius, and the discrete arc-length step. Python is the medium, but the computational argument comes from the model.

[](#fig-phugoid-curves) reproduces the phugoid curves from von Kármán's _Aerodynamics_. He never says _how_ he drew them, but we're guessing by hand, too. We did pretty well!

```{figure} ./figures/vonKarman-phugoids.png
:label: fig-phugoid-curves
:alt: A family of looping and cusped phugoid trajectories
:align: center

Phugoid curves in von Kármán's _Aerodynamics_ [@vonkarman1954, pp. 149–151].
```

:::{warning .simple .dropdown icon=false open=false} Your verdict

Conclude your notebook with a short engineering verdict:

- Which check provides the strongest evidence that your tracer implements the intended model, and why?
- Which plausible defect could have escaped the agent's original checks?
- Which part of your specification did you revise after finding that $C>2/3$ is unreachable from valid initial conditions?
- Which agent suggestions did you accept, reject, or correct?
- What have you verified about the computation, and what remains unverified?

State whether you accept the implementation, accept it with limitations, or require revision. Support that decision with evidence rather than the fact that the code ran or the tests passed. Use the short [evidence record](../../appendices/verification-patterns.md#verification-evidence-record) and leave a lightweight [agent record](../../appendices/agent-use.md#agent-record).
:::

This trammel algorithm marched geometrically along the trajectory in increments of arc length $ds$. In the next lesson, we will derive the differential equation for a small perturbation of the horizontal phugoid and march a dynamical state forward in time using Euler's method. The same computational patterns—state, steps, loops, updates, and inspection—will reappear in a new numerical setting.